In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from churn.config import GOLD_TABLE, ID, TARGET, RANDOM_SEED
from churn.validation import validar_gold
from churn.pipeline import construir_pipeline

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

pipe = construir_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
pipe.fit(X_train, y_train)

In [0]:
y_proba = pipe.predict_proba(X_test)[:, 1]
y_proba

## SPA

### El umbral de decisión

La regresión logística devuelve una probabilidad entre 0 y 1 para cada cliente. El método `predict()` la convierte en un veredicto cortando por 0,5, de modo que todo cliente por encima de ese valor se marca como baja y el resto se da por retenido.

Ese 0,5 tiene una explicación dentro del modelo. Es el punto en el que la combinación lineal de las variables vale cero, la frontera que separa las dos clases. Lo que no tiene es ninguna relación con el negocio.

Bajar el umbral significa marcar a más clientes. Se detectan más bajas reales y al mismo tiempo aumentan los falsos positivos, clientes que no pensaban marcharse y que recibirán una oferta de retención innecesaria. Subirlo hace lo contrario, molesta a menos gente y deja escapar a más.

La hipótesis de partida es que a la empresa le conviene bajarlo. Perder un cliente cuesta bastante más que una llamada de más, así que aceptar falsos positivos a cambio de detectar más bajas debería salir rentable. Queda por cuantificar con las cifras de facturación y esa cuantificación es lo que cierra esta fase.

El trabajo de este notebook es medir ese compromiso a lo largo de todo el rango de umbrales y elegir uno con criterio, junto con la métrica que servirá para comparar modelos en la fase siguiente.

## ENG

### The decision threshold

Logistic regression returns a probability between 0 and 1 for each customer. The `predict()` method turns it into a verdict by cutting at 0.5, so every customer above that value is flagged as churn and the rest are taken as retained.

That 0.5 has an explanation inside the model. It is the point where the linear combination of the variables equals zero, the boundary that separates the two classes. What it does not have is any relationship with the business.

Lowering the threshold means flagging more customers. More real churners are detected and at the same time false positives go up, customers who were not planning to leave and who will get a retention offer they did not need. Raising it does the opposite, it bothers fewer people and lets more of them slip away.

The starting hypothesis is that lowering it suits the company. Losing a customer costs considerably more than one unnecessary call, so accepting false positives in exchange for detecting more churn should pay off. This remains to be quantified with the billing figures and that quantification is what closes this phase.

The work of this notebook is to measure that trade-off across the whole range of thresholds and to choose one with a criterion, along with the metric that will be used to compare models in the next phase.

In [0]:
import seaborn as sns

sns.stripplot(x=y_test, y=y_proba, hue=y_test, alpha=0.4, size=3, legend=False)

plt.axhline(0.5, color="black", linestyle="--")
plt.xlabel("0 = se quedó, 1 = se fue")
plt.ylabel("Probabilidad de baja según el modelo")

In [0]:
from sklearn.metrics import precision_recall_curve

precision, recall, umbrales = precision_recall_curve(y_test, y_proba)

i = np.argmin(np.abs(umbrales - 0.5))

plt.plot(recall, precision, color="#0072B2")
plt.scatter(recall[i], precision[i], color="#E69F00", zorder=5, s=60,
            label=f"umbral 0,5  (recall {recall[i]:.2f}, pprecision {precision[i]:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()

In [0]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, X_train, y_train, cv=5)
print(f"Scores: {scores}")
print(f"Mean: {scores.mean()}")
print(f"Std: {scores.std()}")

In [0]:
# Trabajamos en numpy para no arrastrar el índice de pandas
y_real = y_test.to_numpy()
total_bajas = y_real.sum()

umbrales = np.arange(0.10, 0.75, 0.05)
filas = []

for umbral in umbrales:
    umbral = round(umbral, 2)
    marcados = y_proba >= umbral

    llamadas = marcados.sum()
    detectados = (marcados & (y_real == 1)).sum()
    falsas_alarmas = (marcados & (y_real == 0)).sum()
    perdidos = (~marcados & (y_real == 1)).sum()

    filas.append({
        "umbral": umbral,
        "llamadas": llamadas,
        "detectados": detectados,
        "perdidos": perdidos,
        "falsas_alarmas": falsas_alarmas,
        "precision": detectados / llamadas,
        "recall": detectados / total_bajas,
        "coste_por_captura": falsas_alarmas / detectados,
    })

umbrales_df = pd.DataFrame(filas)

display(umbrales_df)

In [0]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.plot(umbrales_df["umbral"], umbrales_df["detectados"],
         color="#E69F00", marker="o", label="Bajas detectadas")
ax1.plot(umbrales_df["umbral"], umbrales_df["falsas_alarmas"],
         color="#0072B2", marker="o", label="Falsas alarmas")
ax1.invert_xaxis()
ax1.set_xlabel("Umbral (baja hacia la derecha)")
ax1.set_ylabel("Clientes")
ax1.legend()

ax2.plot(umbrales_df["umbral"], umbrales_df["coste_por_captura"],
         color="#0072B2", marker="o")
ax2.invert_xaxis()
ax2.set_xlabel("Umbral (baja hacia la derecha)")
ax2.set_ylabel("Falsas alarmas por cada baja detectada")

plt.tight_layout()

In [0]:
umbrales_df = umbrales_df.sort_values("umbral", ascending=False).reset_index(drop=True)

umbrales_df["detectados_nuevos"] = umbrales_df["detectados"].diff()
umbrales_df["falsas_alarmas_nuevas"] = umbrales_df["falsas_alarmas"].diff()
umbrales_df["coste_marginal"] = umbrales_df["falsas_alarmas_nuevas"] / umbrales_df["detectados_nuevos"]

display(umbrales_df)

In [0]:
umbrales_df[["umbral","detectados_nuevos", "falsas_alarmas_nuevas", "coste_marginal"]]

In [0]:
plt.plot(umbrales_df["umbral"], umbrales_df["detectados_nuevos"],
         color="#E69F00", marker="o", label="Bajas nuevas que pillas")
plt.plot(umbrales_df["umbral"], umbrales_df["falsas_alarmas_nuevas"],
         color="#0072B2", marker="o", label="Falsas alarmas nuevas que pagas")

plt.gca().invert_xaxis()
plt.xlabel("Umbral (baja hacia la derecha)")
plt.ylabel("Clientes nuevos en ese escalón")
plt.legend()

## SPA: 
### Clonclusiones y estudio preliminar de las métricas:

El modelo, como ya hemos dicho, clasifica entre "cliente que se dará de baja" y "cliente que no se dará de baja" de una manera sencilla: Si saca una "puntuación" entre 0.5 y 1, entonces será un cliente que "se dará de baja". Si saca de 0.5 a 0, ese cliente "no se dará de baja".
El problema está en que nada es perfecto y mucho menos en una solución de negocio. Hay clientes cercanos al 0.5 que podrían o no darse de baja. Ese umbral del 0.5 es propio del modelo, pero se puede ajustar para hacer más llamadas y rescatar a mayor número de clientes, a costa de llamar a algunos que no querían darse de baja (falsas alarmas).

### La tabla de umbrales

Recorriendo umbrales de 0,10 a 0,70 se cuenta cuántos clientes caen en cada casilla. Las llamadas, las bajas detectadas y las falsas alarmas crecen todas al bajar la raya del umbral, así que los totales no distinguen un umbral bueno de uno malo. Lo que decide es el reparto entre las dos clases de error.

Comparando escalón a escalón, el que llega a 0,50 es el último en el que se ganan más bajas que falsas alarmas se pagan, con 38 capturas nuevas frente a 24 llamadas de más. A partir de 0,45 las dos series se cruzan y no vuelven a juntarse. Entre 0,45 y 0,30 mantenemos un nivel de captura de bajas estable. Más allá, no conseguimos retomar el ratio captura de bajas - falsa alarma.

### Conclusión

Por debajo de 0,45 cada escalón cuesta más falsas alarmas de las bajas que aporta, de manera que bajar la raya deja de ser un ajuste y pasa a ser una decisión de negocio. Lo que hace interesante el tramo entre 0,40 y 0,30 es que el precio de esa compra se mantiene estable, alrededor de llamada y media por cada baja detectada, mientras que por debajo se dispara.

Elegir un valor concreto dentro de ese tramo es una decisión de negocio y no de modelado. Hacen falta el coste de una oferta de retención, el valor de un cliente que se queda y el porcentaje de ofertas que acaban funcionando. Con esos tres números el cálculo es inmediato. Sin ellos, lo correcto es dejar el rango señalado y la decisión pendiente.

%md
## ENG:
### Conclusions and preliminary study of the metrics:

The model, as already stated, classifies between "customer who will churn" and "customer who will not churn" in a simple way. If it produces a "score" between 0.5 and 1, then that customer will churn. If the score falls between 0.5 and 0, that customer will not churn.

The problem is that nothing is perfect, least of all a business solution. There are customers close to 0.5 who might or might not churn. That 0.5 threshold belongs to the model, but it can be adjusted to make more calls and rescue a larger number of customers, at the cost of calling some who never intended to leave (false alarms).

### The threshold table

Sweeping thresholds from 0.10 to 0.70, we count how many customers fall into each box. Calls, detected churners and false alarms all grow as the threshold line comes down, so the totals do not tell a good threshold from a bad one. What decides is how the two kinds of error are split.

Comparing step by step, the one that reaches 0.50 is the last in which we gain more churners than the false alarms we pay for, with 38 new catches against 24 extra calls. From 0.45 onwards the two series cross and never meet again. Between 0.45 and 0.30 the price stays stable, at around 1.4 to 1.7 false alarms for each churner detected. Beyond that, the catch to false alarm ratio never recovers.

### Conclusion

Below 0.45 each step costs more false alarms than the churners it brings in, so lowering the line stops being an adjustment and becomes a business decision. What makes the stretch between 0.40 and 0.30 interesting is that the price paid stays stable, around one and a half calls for each churner detected, while below it shoots up.

Choosing a specific value within that stretch is a business decision and not a modelling one. It requires the cost of a retention offer, the value of a customer who stays and the percentage of offers that actually work. With those three numbers the calculation is immediate. Without them, the right thing to do is to mark the range and leave the decision pending.